# Experiment 5 — AutoGluon TimeSeries

This experiment uses the shared preprocessing module, then aggregates records into one monthly rent series per `BLOCK + STREET + FLAT_TYPE + FLAT_MODEL` combination. It uses only aggregate COE and stock features as known covariates; individual stock columns are not introduced. Validation forecasts the final observed months without exposing their rent targets.

In [7]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'), Path('/content/drive/MyDrive/KDDM-project')]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

try:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
except ImportError as exc:
    raise ImportError('Install AutoGluon TimeSeries with: pip install autogluon.timeseries') from exc

from preprocessing import preprocess_train_test
RANDOM_STATE = 42
TARGET = 'MONTHLY_RENT'
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
data_dir, datasets, train_df, test_df = preprocess_train_test(project_root / 'data')
train_df['period'] = pd.to_datetime(train_df['RENT_APPROVAL_DATE'] + '-01')
test_df['period'] = pd.to_datetime(test_df['RENT_APPROVAL_DATE'] + '-01')
print(f'Train: {train_df.period.min().date()} to {train_df.period.max().date()}')
print(f'Test: {test_df.period.min().date()} to {test_df.period.max().date()}')

Train: 2021-01-01 to 2025-03-01
Test: 2025-03-01 to 2026-07-01


## Build monthly series and leakage-safe validation

Each item is a property-group combination. The time-series model sees the target only for each item’s historical prefix. Covariates for the forecast horizon come from the corresponding rows but contain no rent target. Short or unseen combinations use a historical-mean fallback.

In [8]:
COVARIATES = [
    'coe_quota_premium_mean', 'coe_bids_mean',
    'stock_close_mean', 'stock_volume_mean',
]
ITEM_ID_COLUMNS = ['BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL']
# The exact property combinations are sparse; keep groups with at least
# one year of observed monthly rents and let the panel retain missing months.
MIN_OBSERVED_MONTHS = 12

def make_item_id(frame):
    values = frame[ITEM_ID_COLUMNS].fillna('Unknown').astype(str)
    return values.agg(' | '.join, axis=1)

train_df['item_id'] = make_item_id(train_df)
test_df['item_id'] = make_item_id(test_df)
monthly = (train_df.groupby(['item_id', 'period'], as_index=False)
    .agg(target=(TARGET, 'mean'), **{c: (c, 'mean') for c in COVARIATES})
    .sort_values(['item_id', 'period']).reset_index(drop=True))
monthly['timestamp'] = monthly['period']

# Use the final 12 calendar months as a chronological forecast test.
VALIDATION_HORIZON = 12
all_periods = pd.date_range(monthly['period'].min(), monthly['period'].max(), freq='MS')
validation_cutoff = all_periods[-VALIDATION_HORIZON]
history_periods = all_periods[all_periods < validation_cutoff]
validation_periods = all_periods[all_periods >= validation_cutoff]
history_counts = monthly[monthly['period'].isin(history_periods)].groupby('item_id')['target'].count()
validation_items = set(monthly.loc[monthly['period'].isin(validation_periods), 'item_id'])
eligible_items = sorted(set(history_counts[history_counts >= MIN_OBSERVED_MONTHS].index) & validation_items)

def make_panel(item_ids, periods, include_target=True, covariate_source=train_df):
    panel = pd.MultiIndex.from_product([item_ids, periods], names=['item_id', 'period']).to_frame(index=False)
    covariates = (covariate_source[covariate_source['period'].isin(periods)]
        .groupby('period', as_index=False)[COVARIATES].mean())
    panel = panel.merge(covariates, on='period', how='left')
    if include_target:
        panel = panel.merge(monthly[['item_id', 'period', 'target']], on=['item_id', 'period'], how='left')
    panel['timestamp'] = panel['period']
    return panel

def to_ts(frame, include_target=True):
    columns = ['item_id', 'timestamp'] + (['target'] if include_target else []) + COVARIATES
    return TimeSeriesDataFrame.from_data_frame(frame[columns], id_column='item_id', timestamp_column='timestamp')

history_monthly = make_panel(eligible_items, history_periods, include_target=True)
validation_monthly = make_panel(eligible_items, validation_periods, include_target=True)
history_ts = to_ts(history_monthly)
validation_future_ts = to_ts(validation_monthly, include_target=False)
print(f'Eligible item series: {len(eligible_items):,} of {monthly.item_id.nunique():,}')
print(f'History months: {len(history_periods)}; validation horizon: {len(validation_periods)}')
print(f'Item identity: {ITEM_ID_COLUMNS}')
print(f'Known covariates: {COVARIATES}')

Eligible item series: 535 of 39,125
History months: 39; validation horizon: 12
Item identity: ['BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL']
Known covariates: ['coe_quota_premium_mean', 'coe_bids_mean', 'stock_close_mean', 'stock_volume_mean']


In [9]:
validation_predictor = TimeSeriesPredictor(
    freq='MS',
    target='target',
    prediction_length=VALIDATION_HORIZON,
    known_covariates_names=COVARIATES,
    eval_metric='RMSE',
    path=str(experiment_dir / 'ag_timeseries_validation'),
)
validation_predictor.fit(
    train_data=history_ts,
    presets='medium_quality',
    time_limit=TIME_LIMIT_SECONDS,
    random_seed=RANDOM_STATE,
    verbosity=2,
)
validation_forecast = validation_predictor.predict(
    history_ts, known_covariates=validation_future_ts
)
forecast_frame = validation_forecast.reset_index()[['item_id', 'timestamp', 'mean']]
actual_frame = validation_monthly.dropna(subset=['target'])[['item_id', 'timestamp', 'target']]
validation_scored = actual_frame.merge(forecast_frame, on=['item_id', 'timestamp'], how='inner')
validation_predictions = validation_scored['mean'].to_numpy()
validation_actual = validation_scored['target'].to_numpy()
validation_metrics = pd.DataFrame([{
    'RMSE': mean_squared_error(validation_actual, validation_predictions) ** 0.5,
    'MAE': mean_absolute_error(validation_actual, validation_predictions),
    'R2': r2_score(validation_actual, validation_predictions),
}])
display(validation_metrics)
print(f'Scored {len(validation_scored):,} observed item-months')
display(validation_predictor.leaderboard(silent=True))

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/Users/metul/Desktop/KDDM-project/exploration/metul/ag_timeseries_validation'
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       5.03 GB / 16.00 GB (31.5%)
Disk Space Avail:   201.45 GB / 460.38 GB (43.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'light',
 'known_covariates_names': ['coe_quota_premium_mean',
                            'coe_bids_mean',
                        

,RMSE,MAE,R2
0,477.592252,367.879829,0.320722


Scored 2,985 observed item-months


,model,score_val,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-628.563856,2.978934,0.172002,8
1,Chronos2,-682.690473,0.998290,4.036683,6
2,Theta,-688.274240,0.123975,0.010657,5
3,ETS,-714.253498,1.502501,0.010645,4
4,Toto2,-738.026889,0.033851,1.796024,7
5,SeasonalNaive,-782.072708,1.772762,0.008515,1
6,RecursiveTabular,-836.823699,0.036339,1.503477,2
7,DirectTabular,-864.890533,0.045152,1.693391,3


## Fit on the pre-test history and forecast test months

The test covariates are averaged by month and shared across item series. Each item’s target history intentionally ends before the first test month to avoid using a training rent average for a month being forecast.

In [10]:
test_start = test_df['period'].min()
test_periods = pd.date_range(test_start, test_df['period'].max(), freq='MS')
final_history_periods = pd.date_range(monthly['period'].min(), test_start - pd.offsets.MonthBegin(1), freq='MS')
test_items = set(test_df['item_id'])
final_counts = monthly[monthly['period'].isin(final_history_periods)].groupby('item_id')['target'].count()
final_eligible_items = sorted(test_items & set(final_counts[final_counts >= MIN_OBSERVED_MONTHS].index))
final_history_monthly = make_panel(final_eligible_items, final_history_periods, include_target=True)
test_monthly = make_panel(final_eligible_items, test_periods, include_target=False, covariate_source=test_df)
test_horizon = len(test_periods)
final_history_ts = to_ts(final_history_monthly)
test_future_ts = to_ts(test_monthly, include_target=False)

final_predictor = TimeSeriesPredictor(
    freq='MS',
    target='target',
    prediction_length=test_horizon,
    known_covariates_names=COVARIATES,
    eval_metric='RMSE',
    path=str(experiment_dir / 'ag_timeseries_final'),
)
final_predictor.fit(
    train_data=final_history_ts,
    presets='medium_quality',
    time_limit=TIME_LIMIT_SECONDS,
    random_seed=RANDOM_STATE,
    verbosity=2,
)
forecast = final_predictor.predict(final_history_ts, known_covariates=test_future_ts)
forecast_by_item_period = forecast.reset_index().set_index(['item_id', 'timestamp'])['mean']
submission_keys = pd.MultiIndex.from_arrays([test_df['item_id'], test_df['period']])
submission = test_df.copy()
submission['PREDICTED_MONTHLY_RENT'] = forecast_by_item_period.reindex(submission_keys).to_numpy()
# Fallback for unseen or short-history combinations: their historical mean,
# followed by the overall pre-test mean if the combination is entirely new.
fallback_source = train_df[train_df['period'] < test_start]
fallback_by_item = fallback_source.groupby('item_id')[TARGET].mean()
fallback = test_df['item_id'].map(fallback_by_item).fillna(fallback_source[TARGET].mean())
submission['PREDICTED_MONTHLY_RENT'] = submission['PREDICTED_MONTHLY_RENT'].fillna(fallback).astype(float).clip(lower=0)
output_path = experiment_dir / 'exp_5_timeseries_predictions.csv'
submission.drop(columns=['period']).to_csv(output_path, index=False)
print(f'Saved {len(submission):,} predictions to {output_path}')
display(submission[['RENT_APPROVAL_DATE', 'TOWN', 'FLAT_TYPE', 'PREDICTED_MONTHLY_RENT']].head())

Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/Users/metul/Desktop/KDDM-project/exploration/metul/ag_timeseries_final'
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.45 GB / 16.00 GB (27.8%)
Disk Space Avail:   201.45 GB / 460.38 GB (43.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'light',
 'known_covariates_names': ['coe_quota_premium_mean',
                            'coe_bids_mean',
                            '

Saved 50,000 predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_5_timeseries_predictions.csv


,RENT_APPROVAL_DATE,TOWN,FLAT_TYPE,PREDICTED_MONTHLY_RENT
0,2025-03,kallang/whampoa,3-room,2820.0
1,2025-03,bukit timah,4-room,3270.0
2,2025-03,pasir ris,4-room,3600.0
3,2025-03,punggol,4-room,3150.0
4,2025-03,tampines,4-room,4050.0
